# 28 - Multi-Robot Coordination

## What / Why / How

**What we are trying to do:** Coordinate several robots with formation control, collision repulsion, and simple task assignment.

**Why this matters:** Multiple robots introduce communication, allocation, and interaction problems that single-robot intuition does not cover.

**How we will do it:** Move a team into formation, add pairwise spacing behavior, and greedily assign robots to task locations.

## Goal

Multi-robot systems add coordination problems:

- Formation control
- Collision avoidance
- Communication limits
- Task allocation

In [ ]:
import math
import random
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see plots: pip install -r requirements.txt")

In [ ]:
from robotics_mastery.safety import velocity_obstacle_safe, limit_norm

rng = np.random.default_rng(28)
positions = rng.uniform(-1.5, 1.5, size=(5, 2))
desired_offsets = np.array([[-1,0], [-0.5,0.5], [0,0], [0.5,0.5], [1,0]], dtype=float)
center_goal = np.array([2.0, 1.0])
routes = [positions.copy()]

for _ in range(120):
    center = positions.mean(axis=0)
    velocities = []
    for i, p in enumerate(positions):
        desired = center_goal + desired_offsets[i]
        v = 0.8 * (desired - p)
        for j, other in enumerate(positions):
            if i == j:
                continue
            diff = p - other
            d = np.linalg.norm(diff)
            if d < 0.45:
                v += 0.2 * diff / (d + 1e-6)
        velocities.append(limit_norm(v, 0.08))
    positions = positions + np.array(velocities)
    routes.append(positions.copy())
routes = np.array(routes)
print("final positions:\n", positions)
print("formation error:", np.linalg.norm((positions - positions.mean(axis=0)) - (desired_offsets - desired_offsets.mean(axis=0))))

In [ ]:
tasks = np.array([[2, 0], [2, 2], [3, 1], [1.5, 1.2], [2.5, -0.5]], dtype=float)
costs = np.linalg.norm(positions[:, None, :] - tasks[None, :, :], axis=2)
assignment = []
remaining = set(range(len(tasks)))
for robot in range(len(positions)):
    best_task = min(remaining, key=lambda t: costs[robot, t])
    assignment.append((robot, best_task, costs[robot, best_task]))
    remaining.remove(best_task)
print("greedy assignment:", assignment)

In [ ]:
if HAS_PLOT:
    plt.figure(figsize=(6, 4))
    for i in range(routes.shape[1]):
        plt.plot(routes[:, i, 0], routes[:, i, 1], label=f"robot {i}")
    plt.scatter(tasks[:, 0], tasks[:, 1], marker="x", c="black", label="tasks")
    plt.axis("equal")
    plt.grid(True, alpha=0.3)
    plt.legend(ncol=2, fontsize=8)
    plt.show()
else:
    plot_unavailable()

## Exercises

1. Add communication dropout.
2. Replace greedy assignment with Hungarian matching after installing SciPy.
3. Add velocity obstacle checks between robot pairs.